In [2]:
%pip install nvcc4jupyter -U --no-cache-dir --force-reinstall
%load_ext nvcc4jupyter

Note: you may need to restart the kernel to use updated packages.
Source files will be saved in "/tmp/tmpm5mz89j8".


In [10]:
%%cuda
#include "cuda_runtime.h"
#include "device_launch_parameters.h"
#include <stdio.h>


int _ConvertSMVer2Cores(const int major, const int minor) {
    switch ((major << 4) + minor) {
    case 0x10: case 0x11: case 0x12: case 0x13:
        return 8;
    case 0x20:
        return 32;
    case 0x21: case 0x30: case 0x35: case 0x37:
        return 192;
    case 0x50: case 0x52: case 0x53:
        return 128;
    case 0x60:
        return 64;
    case 0x61: case 0x62:
        return 128;
    case 0x70: case 0x72: case 0x75:
        return 64;
    case 0x80: case 0x86: case 0x87: case 0x89: case 0x8A:
        return 64;
    case 0x90:
        return 128;
    case 0xC0: case 0xC1: case 0xC2: case 0xC3: // Compute capability 12.x (Blackwell)
        return 128;
    default:
        printf("Unknown compute capability %d.%d\n", major, minor);
        return -1;
    }
}


cudaError_t cardProperties() {
    cudaError_t cudaStatus = cudaSuccess;
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);

    printf("Number of CUDA devices: %d\n", deviceCount);

    for (int dev = 0; dev < deviceCount; dev++) {
        cudaDeviceProp deviceProp;
        cudaGetDeviceProperties(&deviceProp, dev);

        printf("\nDevice %d: %s\n", dev, deviceProp.name);
        printf("Compute Capability: %d.%d\n", deviceProp.major, deviceProp.minor);
        printf("Number of SMs: %d\n", deviceProp.multiProcessorCount);
        printf("Number of SP per SM: %d\n", _ConvertSMVer2Cores(deviceProp.major, deviceProp.minor));
        printf("Total CUDA Cores: %d\n", deviceProp.multiProcessorCount * _ConvertSMVer2Cores(deviceProp.major, deviceProp.minor));
        printf("Max Threads per Block: %d\n", deviceProp.maxThreadsPerBlock);
        printf("Total registers per block: %d\n", deviceProp.regsPerBlock);
        printf("Warp size: %d\n", deviceProp.warpSize);
        printf("\n=== Memory Information ===\n");
        printf("Total shared memory per block: %zu bytes\n", deviceProp.sharedMemPerBlock);
        printf("Total global memory: %zu bytes (%.2f GB)\n",
               deviceProp.totalGlobalMem,
               deviceProp.totalGlobalMem / 1024.0 / 1024.0 / 1024.0);
        printf("Total constant memory: %zu bytes\n", deviceProp.totalConstMem);

        // Query memory bandwidth using cudaDeviceGetAttribute
        int memClockRate = 0;
        int memBusWidth = 0;

        cudaStatus = cudaDeviceGetAttribute(&memClockRate, cudaDevAttrMemoryClockRate, dev);
        if (cudaStatus == cudaSuccess) {
            cudaStatus = cudaDeviceGetAttribute(&memBusWidth, cudaDevAttrGlobalMemoryBusWidth, dev);
            if (cudaStatus == cudaSuccess) {
                // Memory bandwidth formula: 2 * clock_rate (kHz) * (bus_width / 8) / 1e6 = GB/s
                double bandwidth = 2.0 * memClockRate * (memBusWidth / 8) / 1.0e6;
                printf("Memory Clock Rate: %d kHz\n", memClockRate);
                printf("Memory Bus Width: %d bits\n", memBusWidth);
                printf("Peak Memory Bandwidth: %.2f GB/s\n", bandwidth);
            } else {
                printf("Memory bandwidth: Unable to query (cudaDevAttrGlobalMemoryBusWidth not available)\n");
            }
        } else {
            printf("Memory bandwidth: Unable to query (cudaDevAttrMemoryClockRate not available)\n");
        }
    }
    return cudaSuccess;
}


int main(int argc, char** argv){
    cudaError_t cudaStatus = cardProperties();

    if (cudaStatus != cudaSuccess) {
        fprintf(stderr, "cardProperties failed!");
        return 1;
    }

    cudaStatus = cudaDeviceReset();
    if (cudaStatus != cudaSuccess) {
        fprintf(stderr, "cudaDeviceReset failed!");
        return 1;
    }

    return 0;
}

Number of CUDA devices: 1

Device 0: NVIDIA GeForce RTX 5060 Ti
Compute Capability: 12.0
Number of SMs: 36
Number of SP per SM: 128
Total CUDA Cores: 4608
Max Threads per Block: 1024
Total registers per block: 65536
Warp size: 32

=== Memory Information ===
Total shared memory per block: 49152 bytes
Total global memory: 8077180928 bytes (7.52 GB)
Total constant memory: 65536 bytes
Memory Clock Rate: 14001000 kHz
Memory Bus Width: 128 bits
Peak Memory Bandwidth: 448.03 GB/s



In [9]:
!ncu --version

NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2024 NVIDIA Corporation
Version 2024.2.1.0 (build 34372528) (public-release)
